## Imports

In [145]:
import sklearn
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib as mpl
import scipy.io as sio
import itertools

In [146]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using mps device


## Linear Autoencoders

In [147]:
class SCA_encoder(nn.Module):
    '''
    Input: a (T * N) matrix X, of N neurons recorded over T timesteps.
    Output: an affine mapping of X to K-dimensional space.
    '''
    def __init__(
        self,
        N: 100, # number of neurons
        K: 3, # dimensionality
        Q: None # Initalization params
    ):
        super().__init__()
        self.U = nn.Linear(N,K)
        if Q != None:
            Q = nn.Parameter(Q)
            self.U.weight = Q
        nn.init.constant_(self.U.bias,0)

    def forward(self,x):
        encoded_x = self.U(x)
        return encoded_x

class SCA_decoder(nn.Module):
    '''
    Input: affine mapped X in K-dimensional space
    Output: X_hat, or the reconstructed X back in neural activity space
    '''
    def __init__(
        self,
        N: 100, # number of neurons
        K: 3, # dimensionality
        Q: None # Initialization params
    ):
        super().__init__()
        self.V = nn.Linear(K,N)
        if Q != None:
            QT = nn.Parameter(Q.transpose(1,0))
            self.V.weight = QT
        nn.init.constant_(self.V.bias,0)

    def forward(self, encoded_x):
        normalized_V_weight = nn.functional.normalize(self.V.weight.data,dim=1)
        self.V.weight.data = normalized_V_weight
        x_hat = self.V(encoded_x)
        return x_hat
        

## Training Function

In [148]:
def train(X, encoder, decoder, optimizer, lambda_sparse = 0, lambda_orth = 0):

    X = X.to(device)

    # Calculate reconstruction loss
    encoded_X = encoder(X)
    decoded_X = decoder(encoded_X)
    reconstruction_loss = torch.norm(X - decoded_X)**2

    # Calculate sparsity penalty
    sparsity_loss = lambda_sparse * torch.norm(encoded_X)

    # Calculate orthogonality penalty
    V_weight = nn.functional.normalize(decoder.V.weight.data,dim=1)
    VTV = torch.matmul(V_weight.transpose(1,0),V_weight)
    I = torch.eye(VTV.size(dim=0),device=device)
    orth_loss = lambda_orth * torch.norm(VTV - I)**2

    # Sum to calculate loss
    loss = reconstruction_loss + sparsity_loss + orth_loss

    # Backpropagate

    loss.backward()

    optimizer.step()
    optimizer.zero_grad()

    return loss.item()


## Training Loop

In [149]:
def training_loop(X, encoder, decoder, lambda_sparse = 0, lambda_orth = 0, epochs=100):

    lr = 1e-3
    min_lr = 5e-4
    plateau_len = 100
    factor = 0.5
    all_params = itertools.chain(encoder.parameters(), decoder.parameters())
    optimizer = torch.optim.Adam(all_params,lr=lr)
    lr_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=factor, patience=plateau_len, min_lr=min_lr)
    
    losses = torch.zeros(epochs)
    for t in range(epochs):
        loss_t = train(X, encoder, decoder, optimizer, lambda_sparse, lambda_orth)
        losses[t] = loss_t
        if t % 100 == 0:
            print(f"Epoch {t+1}\n-------------------------------")
            print(f"loss: {loss_t:>7f}  [{t:>5d}/{epochs:>5d}]")
        if t % 1000 == 0:
            current_lr = lr_scheduler.get_last_lr()[-1]
            print(f"learning rate: {current_lr:>7f}")
        lr_scheduler.step(loss_t)
    
    return losses

## Sample-Weighting Matrix and PCA

In [150]:
def create_W(X):
    '''
    Input: T x N matrix X of T timesteps of N neurons' neural activity
    Output: Diagonal T x T matrix W, where each entry contains the inverse sum-square at that timestep
    '''
    sum_sq_activity = torch.sum(X,axis=1)**2
    inverse = 1/ sum_sq_activity
    W = torch.diag(inverse)
    return W
    

### Blueprint for weight initialization

In [151]:
N = 10
K = 3
T = 15

data_ex = torch.randn(T,N)

# Sample_weight
Weight_ex = create_W(data_ex)

# Find the Vh Matrix
U,S,Vh = torch.linalg.svd(torch.matmul(Weight_ex,data_ex))

# Take the first K rows of Vh and transpose into the N x K matrix Q
Q = Vh[0:K].transpose(1,0)

## Blueprint for hyperparameter initialization

In [152]:
QQT = torch.matmul(Q,Q.transpose(1,0))

init_reconstruction = torch.norm(torch.matmul(Weight_ex,(data_ex - (torch.matmul(data_ex,QQT)))))**2
init_orth = torch.tensor(0.1).expand(K,K)-torch.eye(K)*0.1
init_orth = torch.norm(init_orth)**2
init_sparse = torch.norm(torch.matmul(data_ex,Q))

lambda_init_orth = 0.1*init_reconstruction / init_orth
lambda_init_sparse = 0.1*init_reconstruction / init_sparse

print(lambda_init_orth)
print(lambda_init_sparse)

tensor(74.5651)
tensor(0.5017)


## Writing the Final Wrapper Function

In [159]:
def SCA(X,K,epochs = 100000):

    #---Sub-functions---#
    def create_W(X):
        '''
        Input: T x N matrix X of T timesteps of N neurons' neural activity
        Output: Diagonal T x T matrix W, where each entry contains the inverse sum-square at that timestep
        '''
        sum_sq_activity = torch.sum(X,axis=1)**2
        inverse = 1/ sum_sq_activity
        W = torch.diag(inverse)
        return W
    
    # Extract T and N from data tensor
    T,N = X.shape
    
    #---Determining Initialization Parameters and Hyperparameters---#

    # Sample_weight
    W = create_W(X)

    # Find Vh
    U,S,Vh = torch.linalg.svd(torch.matmul(W,X))

    # Find Q
    Q = Vh[0:K].transpose(1,0)

    # UV = QQT
    QQT = torch.matmul(Q,Q.transpose(1,0))

    # Calculate initialized reconstruction cost
    init_reconstruction = torch.norm(torch.matmul(W,(X - (torch.matmul(X,QQT)))))**2

    # Calculate initialized orthogonality cost if off-diagonal entries of VVT = 0.1
    # Since V is orthonormal, VVT - I is 0.1 everywhere except for diagonal, which is 0
    init_orth = torch.tensor(0.1).expand(K,K)-torch.eye(K)*0.1
    init_orth = torch.norm(init_orth)**2

    # Calculate initialized sparsity cost
    init_sparse = torch.norm(torch.matmul(X,Q))

    # Algebraically rearrange to determine initalization lambdas
    lambda_init_orth = 0.1*init_reconstruction / init_orth
    lambda_init_sparse = 0.1*init_reconstruction / init_sparse

    #---Initializing the autoencoders---#
    
    QT = Q.transpose(1,0)
    encoder = SCA_encoder(N=N,K=K,Q=QT).to(device)
    decoder = SCA_decoder(N=N,K=K,Q=QT).to(device)


    #---Train the autoencoders---#

    losses = training_loop(X=X,encoder=encoder,decoder=decoder,lambda_sparse = lambda_init_sparse,lambda_orth=lambda_init_orth,epochs=epochs)

    #---Extract trained encoder-decoder state_dict---#
    encoder_state_dict = encoder.state_dict()
    decoder_state_dict = decoder.state_dict()

    return {
        'encoder_state_dict':encoder_state_dict,
        'decoder_state_dict':decoder_state_dict,
        'losses':losses
    }


In [160]:
N = 10
K = 3
T = 15

final_test_data = torch.randn(T,N)

SCA(final_test_data,K)

Epoch 1
-------------------------------
loss: 360.819702  [    0/100000]
learning rate: 0.001000
Epoch 101
-------------------------------
loss: 401.924561  [  100/100000]
Epoch 201
-------------------------------
loss: 426.472717  [  200/100000]
Epoch 301
-------------------------------
loss: 451.449341  [  300/100000]
Epoch 401
-------------------------------
loss: 479.658081  [  400/100000]
Epoch 501
-------------------------------
loss: 512.514282  [  500/100000]
Epoch 601
-------------------------------
loss: 546.705383  [  600/100000]
Epoch 701
-------------------------------
loss: 576.490967  [  700/100000]
Epoch 801
-------------------------------
loss: 597.788574  [  800/100000]
Epoch 901
-------------------------------
loss: 609.364014  [  900/100000]
Epoch 1001
-------------------------------
loss: 612.080261  [ 1000/100000]
learning rate: 0.000500
Epoch 1101
-------------------------------
loss: 608.540039  [ 1100/100000]
Epoch 1201
-------------------------------
loss: 602

KeyboardInterrupt: 